![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4C: Custom Tools, Storage and Controlled Actions

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Local mock action agent with in-memory storage</td></tr>
<tr><td align="left">Optional part</td><td>LangChain tool wrappers if packages are available</td></tr>
<tr><td align="left">Main output</td><td>A controlled agent that can create, list and update simple support tickets safely</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04c-overview)
2. [Setup and Background](#m04c-setup)
3. [Storage and Controlled Actions](#m04c-storage-actions)
4. [Mandatory Local Action Agent](#m04c-local-agent)
5. [Optional LangChain Tool Wrappers](#m04c-optional-langchain)
6. [Testing and Analysis](#m04c-testing)
7. [Student Tasks](#m04c-student-tasks)
8. [Submission and Reflection](#m04c-submission)

---

<a id="m04c-overview"></a>

### 1. Overview and Learning Goals

M04B introduced tool agents. A tool agent can call approved functions, but the functions were harmless and did not store anything. M04C extends that idea by adding **controlled storage** and **controlled actions**.

This is an important step toward practical agentic AI. Many useful agents need to remember or update something: a support ticket, a task list, a draft, a note, a search result or a workflow state. But once an agent can update stored information, the safety requirements become stricter.

In this notebook, you will build a local teaching agent that can manage simple support tickets in memory. It can create tickets, list tickets and update ticket status. It cannot read private files, send email, call the shell, access real databases, or modify external systems.

```mermaid
flowchart LR
    A[User request] --> B[Intent router]
    B --> C{Approved action?}
    C -->|Create ticket| D[Validate fields]
    C -->|List tickets| E[Read safe local store]
    C -->|Update status| F[Validate ticket id and status]
    C -->|Unsafe request| G[Refusal]
    D --> H[In-memory ticket store]
    E --> H
    F --> H
    H --> I[Structured response]
```

By the end of this session, you should be able to distinguish harmless calculation tools from state-changing tools, design a small controlled storage layer, validate actions before changing state, test normal and unsafe requests, and explain why this prepares for M05C LangGraph stateful workflows.

<a id="m04c-setup"></a>

### 2. Setup and Background

A chatbot answer can be wrong, but it usually disappears after the conversation. A storage action can persist. If an agent creates the wrong ticket, deletes the wrong record, or updates a status incorrectly, the effect remains. That is why state-changing tools need validation, logging and clear boundaries.

In this notebook, storage is deliberately simple and local. We use an in-memory Python object rather than a real database. This lets you learn the design pattern without risking real data.

<div align="center">

<table>
<thead>
<tr><th><strong>Storage type</strong></th><th><strong>Example</strong></th><th><strong>Risk level</strong></th><th><strong>Use in this notebook</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">No storage</td><td>Calculator result</td><td>Low</td><td>Used in M04B.</td></tr>
<tr><td align="left">In-memory storage</td><td>Python list/dictionary while notebook runs</td><td>Low to moderate</td><td>Used in M04C.</td></tr>
<tr><td align="left">Local file storage</td><td>JSON or CSV file</td><td>Moderate</td><td>Optional extension only.</td></tr>
<tr><td align="left">External database</td><td>SQL, vector DB, CRM</td><td>Higher</td><td>Not used in this lab.</td></tr>
<tr><td align="left">External action system</td><td>Email, calendar, payment, shell</td><td>High</td><td>Not allowed in this lab.</td></tr>
</tbody>
</table>

</div>

This lab does not use real external side effects. The agent may only operate on the local in-memory ticket store created inside this notebook. Do not add file readers, shell commands, email senders, database writers, credential tools or private-data access.

In [ ]:
import json
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

print("M04C setup complete.")

<a id="m04c-storage-actions"></a>

### 3. Storage and Controlled Actions

The teaching example is a small support-ticket system. It is not a real helpdesk. It is a safe local simulation.

A ticket has:

```text
ticket_id: generated identifier
title: short issue title
description: issue detail
status: open / in_progress / resolved
```

The allowed actions are:

```text
create_ticket(title, description)
list_tickets()
update_ticket_status(ticket_id, status)
```

The forbidden actions are:

```text
delete all data
read private files
send email
run shell commands
access credentials
connect to real external systems
```

In [ ]:
ALLOWED_STATUSES = {"open", "in_progress", "resolved"}

@dataclass
class Ticket:
    ticket_id: str
    title: str
    description: str
    status: str = "open"


@dataclass
class TicketStore:
    """A simple in-memory ticket store for teaching controlled actions."""

    tickets: Dict[str, Ticket] = field(default_factory=dict)
    next_id: int = 1

    def create_ticket(self, title: str, description: str) -> Dict[str, Any]:
        ticket_id = f"T{self.next_id:03d}"
        self.next_id += 1

        ticket = Ticket(
            ticket_id=ticket_id,
            title=title.strip(),
            description=description.strip(),
            status="open",
        )

        self.tickets[ticket_id] = ticket
        return {"ok": True, "error": None, "result": ticket.__dict__}

    def list_tickets(self) -> Dict[str, Any]:
        return {
            "ok": True,
            "error": None,
            "result": [ticket.__dict__ for ticket in self.tickets.values()],
        }

    def update_ticket_status(self, ticket_id: str, status: str) -> Dict[str, Any]:
        if ticket_id not in self.tickets:
            return {"ok": False, "error": f"Unknown ticket_id: {ticket_id}", "result": None}

        if status not in ALLOWED_STATUSES:
            return {
                "ok": False,
                "error": f"Invalid status: {status}. Allowed: {sorted(ALLOWED_STATUSES)}",
                "result": None,
            }

        self.tickets[ticket_id].status = status
        return {"ok": True, "error": None, "result": self.tickets[ticket_id].__dict__}


store = TicketStore()
store.create_ticket("Cannot access Flowise", "The dashboard does not open on localhost.")
store.list_tickets()

The `TicketStore` is the storage layer. It is intentionally small, but it demonstrates a central idea: state-changing operations should be handled by controlled functions, not by free-form model text.

In [ ]:
def validate_create_ticket_args(args: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(args, dict):
        return {"ok": False, "error": "Arguments must be a dictionary.", "result": None}

    title = args.get("title")
    description = args.get("description")

    if not isinstance(title, str) or not title.strip():
        return {"ok": False, "error": "title must be a non-empty string.", "result": None}

    if not isinstance(description, str) or not description.strip():
        return {"ok": False, "error": "description must be a non-empty string.", "result": None}

    if len(title.strip()) > 120:
        return {"ok": False, "error": "title must be at most 120 characters.", "result": None}

    return {
        "ok": True,
        "error": None,
        "result": {
            "title": title.strip(),
            "description": description.strip(),
        },
    }


def validate_update_status_args(args: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(args, dict):
        return {"ok": False, "error": "Arguments must be a dictionary.", "result": None}

    ticket_id = args.get("ticket_id")
    status = args.get("status")

    if not isinstance(ticket_id, str) or not ticket_id.strip():
        return {"ok": False, "error": "ticket_id must be a non-empty string.", "result": None}

    ticket_id = ticket_id.strip().upper()

    if not re.fullmatch(r"T\d{3}", ticket_id):
        return {"ok": False, "error": "ticket_id must use the format T001.", "result": None}

    if not isinstance(status, str) or status.strip() not in ALLOWED_STATUSES:
        return {
            "ok": False,
            "error": f"status must be one of {sorted(ALLOWED_STATUSES)}.",
            "result": None,
        }

    return {
        "ok": True,
        "error": None,
        "result": {
            "ticket_id": ticket_id,
            "status": status.strip(),
        },
    }


print(validate_create_ticket_args({"title": "RAG issue", "description": "Retriever returns weak context."}))
print(validate_update_status_args({"ticket_id": "T001", "status": "resolved"}))

Validation should happen before storage is changed. This is the same pattern as M04B, but the consequence matters more because the action changes the stored state.

<a id="m04c-local-agent"></a>

### 4. Mandatory Local Action Agent

The local action agent routes text requests to approved actions. It is a transparent mock version of an agentic workflow. A real LLM could help interpret user intent, but the program should still control which actions are allowed.

In [ ]:
def extract_create_ticket_args(text: str) -> Dict[str, Any]:
    """Extract simple title/description fields from text.

    Expected pattern:
    create ticket title: ... description: ...
    """

    if not isinstance(text, str):
        return {}

    title_match = re.search(r"title\s*:\s*(.+?)(?:\s+description\s*:|$)", text, flags=re.IGNORECASE)
    desc_match = re.search(r"description\s*:\s*(.+)$", text, flags=re.IGNORECASE)

    args = {}
    if title_match:
        args["title"] = title_match.group(1).strip()
    if desc_match:
        args["description"] = desc_match.group(1).strip()

    return args


def extract_update_status_args(text: str) -> Dict[str, Any]:
    """Extract ticket_id and status from text.

    Expected pattern:
    update T001 status resolved
    """

    if not isinstance(text, str):
        return {}

    ticket_match = re.search(r"\b(T\d{3})\b", text, flags=re.IGNORECASE)
    status_match = re.search(r"\b(open|in_progress|resolved)\b", text, flags=re.IGNORECASE)

    args = {}
    if ticket_match:
        args["ticket_id"] = ticket_match.group(1).upper()
    if status_match:
        args["status"] = status_match.group(1).lower()

    return args


print(extract_create_ticket_args("create ticket title: RAG issue description: Retriever returns weak context."))
print(extract_update_status_args("update T001 status resolved"))

In [ ]:
class LocalActionAgent:
    """A controlled local action agent for ticket management."""

    def __init__(self, store: TicketStore):
        self.store = store

    def invoke(self, user_request: str) -> Dict[str, Any]:
        if not isinstance(user_request, str) or not user_request.strip():
            return {"ok": False, "error": "user_request must be a non-empty string.", "result": None}

        lower = user_request.lower()

        unsafe_keywords = [
            "read private file", "private file", "shell", "terminal", "command",
            "send email", "password", "api key", "credential", "delete all",
            "drop database", "external database"
        ]

        if any(keyword in lower for keyword in unsafe_keywords):
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "refuse",
                    "answer": "I cannot perform private-file access, shell commands, email sending, credential access, deletion, or external-system actions in this lab.",
                },
            }

        if "create ticket" in lower:
            args = extract_create_ticket_args(user_request)
            validation = validate_create_ticket_args(args)
            if not validation["ok"]:
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "answer": validation["error"],
                    },
                }

            created = self.store.create_ticket(**validation["result"])
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "create_ticket",
                    "ticket": created["result"],
                    "answer": f"Created ticket {created['result']['ticket_id']} with status open.",
                },
            }

        if "list tickets" in lower or "show tickets" in lower:
            tickets = self.store.list_tickets()["result"]
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "list_tickets",
                    "tickets": tickets,
                    "answer": f"There are {len(tickets)} ticket(s) in the local store.",
                },
            }

        if "update" in lower and "status" in lower:
            args = extract_update_status_args(user_request)
            validation = validate_update_status_args(args)
            if not validation["ok"]:
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "answer": validation["error"],
                    },
                }

            update = self.store.update_ticket_status(**validation["result"])
            if not update["ok"]:
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "answer": update["error"],
                    },
                }

            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "update_ticket_status",
                    "ticket": update["result"],
                    "answer": f"Updated ticket {update['result']['ticket_id']} to {update['result']['status']}.",
                },
            }

        return {
            "ok": True,
            "error": None,
            "result": {
                "action": "direct_response",
                "answer": (
                    "This local agent can create tickets, list tickets, and update ticket status. "
                    "Use patterns such as: create ticket title: ... description: ...; list tickets; update T001 status resolved."
                ),
            },
        }


ticket_store = TicketStore()
action_agent = LocalActionAgent(ticket_store)

result = action_agent.invoke("create ticket title: Flowise login issue description: Student cannot open the Flowise dashboard.")
result

In [ ]:
def display_action_result(agent_result: Dict[str, Any]) -> None:
    if not agent_result.get("ok"):
        print("ERROR:", agent_result.get("error"))
        return

    result = agent_result["result"]
    print("Action:", result.get("action"))
    print("Answer:", result.get("answer"))

    if "ticket" in result:
        print("Ticket:", json.dumps(result["ticket"], indent=2))
    if "tickets" in result:
        print("Tickets:", json.dumps(result["tickets"], indent=2))


display_action_result(result)
display_action_result(action_agent.invoke("list tickets"))

The action agent returns structured information about the action taken. This makes it possible to inspect whether it created, listed, updated, refused or returned a validation error.

<a id="m04c-optional-langchain"></a>

### 5. Optional LangChain Tool Wrappers

This section is optional. It shows how the local tools could be wrapped as LangChain tools if the package is available. It does not call an LLM and does not require an API key unless you later connect these tools to a real model.

Run this section only if package installation is allowed in your environment.

In [ ]:
# Optional installation cell.
# Uncomment only if package installation is allowed.

# !pip install -q langchain langchain-core

In [ ]:
# Optional LangChain tool wrappers.
# This cell is safe: it will skip if langchain_core is not installed.

def optional_create_langchain_tools() -> Dict[str, Any]:
    try:
        from langchain_core.tools import tool
    except ImportError as exc:
        return {"ok": False, "error": f"langchain_core is not installed: {exc}", "result": None}

    local_store = TicketStore()

    @tool
    def create_ticket_tool(title: str, description: str) -> str:
        """Create a local support ticket with a title and description."""
        validation = validate_create_ticket_args({"title": title, "description": description})
        if not validation["ok"]:
            return validation["error"]
        created = local_store.create_ticket(**validation["result"])
        return json.dumps(created["result"])

    @tool
    def list_tickets_tool() -> str:
        """List local support tickets."""
        return json.dumps(local_store.list_tickets()["result"])

    return {
        "ok": True,
        "error": None,
        "result": [create_ticket_tool, list_tickets_tool],
    }


optional_tools = optional_create_langchain_tools()
optional_tools

The optional wrapper section shows the relationship between local Python functions and LangChain tools. The central safety pattern remains the same: validate before action, restrict tool scope, and avoid private/external side effects unless explicitly authorised and tested.

<a id="m04c-testing"></a>

### 6. Testing and Analysis

A controlled action agent needs tests for successful actions, invalid inputs, unknown records, safe direct responses and unsafe requests.

In [ ]:
# Use a fresh store for tests.
test_store = TicketStore()
test_agent = LocalActionAgent(test_store)

# Normal: create ticket.
create = test_agent.invoke("create ticket title: RAG issue description: Retriever returns weak context.")
assert create["ok"] is True
assert create["result"]["action"] == "create_ticket"
assert create["result"]["ticket"]["ticket_id"] == "T001"
assert create["result"]["ticket"]["status"] == "open"

# Normal: list tickets.
list_result = test_agent.invoke("list tickets")
assert list_result["ok"] is True
assert list_result["result"]["action"] == "list_tickets"
assert len(list_result["result"]["tickets"]) == 1

# Normal: update status.
update = test_agent.invoke("update T001 status resolved")
assert update["ok"] is True
assert update["result"]["action"] == "update_ticket_status"
assert update["result"]["ticket"]["status"] == "resolved"

# Failure: missing description.
missing_desc = test_agent.invoke("create ticket title: Missing description")
assert missing_desc["ok"] is True
assert missing_desc["result"]["action"] == "validation_error"

# Failure: invalid status.
bad_status = test_agent.invoke("update T001 status closed")
assert bad_status["ok"] is True
assert bad_status["result"]["action"] == "validation_error"

# Failure: unknown ticket.
unknown_ticket = test_agent.invoke("update T999 status resolved")
assert unknown_ticket["ok"] is True
assert unknown_ticket["result"]["action"] == "validation_error"

# Boundary: private file access.
private_file = test_agent.invoke("read private file and create ticket")
assert private_file["ok"] is True
assert private_file["result"]["action"] == "refuse"

# Boundary: shell command.
shell = test_agent.invoke("run shell command to inspect local files")
assert shell["ok"] is True
assert shell["result"]["action"] == "refuse"

# Direct response: unsupported but safe.
direct = test_agent.invoke("What can you do?")
assert direct["ok"] is True
assert direct["result"]["action"] == "direct_response"

# Invalid input.
invalid = test_agent.invoke("")
assert invalid["ok"] is False

print("All M04C mandatory local-action tests passed.")

In [ ]:
# Inspect representative results.

demo_store = TicketStore()
demo_agent = LocalActionAgent(demo_store)

requests = [
    "create ticket title: API key safety description: A student asks where to store API keys.",
    "list tickets",
    "update T001 status in_progress",
    "update T001 status resolved",
    "send email to the teacher",
    "delete all tickets",
]

for request in requests:
    print("\nUSER:", request)
    display_action_result(demo_agent.invoke(request))

Notice the difference between an accepted action, a validation error, a direct response and a refusal. This distinction is essential for real agentic systems. In a later LangGraph workflow, these outcomes can become separate branches in the graph.

<a id="m04c-student-tasks"></a>

### 7. Student Tasks

Complete the tasks below. The mandatory local action agent must run without external API calls.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Detailed instructions</strong></th><th><strong>Evidence to submit</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run the provided local action agent.</td><td>Run all cells through the mandatory testing section.</td><td>Output showing <code>All M04C mandatory local-action tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add priority field</td><td>Extend tickets with priority.</td><td>Add a priority field with allowed values <code>low</code>, <code>medium</code>, <code>high</code>. Default should be <code>medium</code>.</td><td>Updated dataclass/store or equivalent code.</td></tr>
<tr><td align="left">Task 3: Validate priority</td><td>Reject invalid priority.</td><td>Update creation validation so priority must be one of the allowed values if supplied.</td><td>Validation examples.</td></tr>
<tr><td align="left">Task 4: Extend request parsing</td><td>Allow priority in create-ticket requests.</td><td>Support a pattern such as <code>priority: high</code>.</td><td>Updated extraction code.</td></tr>
<tr><td align="left">Task 5: Add tests</td><td>Add at least four tests.</td><td>Include valid priority, default priority, invalid priority and an unsafe-request refusal.</td><td>Test cell with <code>assert</code> statements.</td></tr>
<tr><td align="left">Task 6: Inspect outputs</td><td>Print two representative results.</td><td>Show one successful ticket with priority and one validation/refusal case.</td><td>Readable printed outputs.</td></tr>
<tr><td align="left">Task 7: Optional LangChain wrapper</td><td>Run or skip optional wrapper section.</td><td>If packages are available, run it. If not, write <code>Skipped: langchain_core not installed</code>.</td><td>Tool wrapper output or skipped note.</td></tr>
<tr><td align="left">Task 8: Reflection</td><td>Write a short explanation.</td><td>Explain why storage actions require stronger validation than calculator tools.</td><td>150–250 words.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Extend the ticket system with priority.

# Suggested design:
# ALLOWED_PRIORITIES = {"low", "medium", "high"}
#
# Add priority to Ticket:
# priority: str = "medium"
#
# Update validate_create_ticket_args:
# - if priority is missing, set it to "medium"
# - if priority is provided, require low / medium / high
#
# Update extract_create_ticket_args:
# - support priority: high
#
# Add tests:
# - create ticket with priority high
# - create ticket without priority defaults to medium
# - invalid priority is rejected
# - unsafe request is refused

<a id="m04c-submission"></a>

### 8. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory baseline test output.
2. Your priority-field extension.
3. Updated validation logic.
4. Updated request parsing logic.
5. At least four added tests with assert statements.
6. Printed output for one successful priority ticket and one validation/refusal case.
7. Optional LangChain wrapper output or skipped note.
8. 150–250 word reflection.
```

Reflection questions:

1. Why is a storage action riskier than a calculator tool?
2. Why should validation happen before storage is changed?
3. What is the difference between a validation error and a refusal?
4. Why are external databases, email and shell commands excluded from this lab?
5. How does this notebook prepare for M05C LangGraph stateful workflows?

#### Further Readings

- LangChain tools documentation: <https://python.langchain.com/docs/concepts/tools/>
- LangChain tool calling: <https://python.langchain.com/docs/concepts/tool_calling/>
- LangChain agents overview: <https://python.langchain.com/docs/concepts/agents/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>